In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
import os

model_path = r"C:\Users\91739\Downloads\AnimeGANv2-master (1)\AnimeGANv2-master\pb_and_onnx_model\shinkai_53.onnx"

image_path = input("Enter image path (example: test.jpg): ").strip().strip('"')

print("Checking files...")

if not os.path.exists(model_path):
    print(" Model path:", model_path)
    raise FileNotFoundError(" Model file not found!")

if not os.path.exists(image_path):
    print("Image path:", image_path)
    raise FileNotFoundError("Image not found! Check path.")

print("Files found")

print("Loading model...")
session = ort.InferenceSession(model_path)
print("Model loaded")

def preprocess(img):
    img = cv2.resize(img, (256, 256))
    img = img.astype(np.float32) / 127.5 - 1
    img = np.expand_dims(img, axis=0)   # (1, 256, 256, 3)
    return img


def postprocess(output):
    output = np.squeeze(output)   # (256, 256, 3)
    output = (output + 1) * 127.5
    output = np.clip(output, 0, 255)
    return output.astype(np.uint8)

print(" Reading image...")
img = cv2.imread(image_path)

print("Image loaded:", img is not None)

if img is None:
    raise ValueError("Failed to load image.")

print(" Converting to anime...")

input_tensor = preprocess(img)

output = session.run(None, {
    session.get_inputs()[0].name: input_tensor
})[0]

anime = postprocess(output)

print("Conversion done!")

cv2.namedWindow("Original Image", cv2.WINDOW_NORMAL)
cv2.namedWindow("Anime Image", cv2.WINDOW_NORMAL)

cv2.imshow("Original Image", img)
cv2.imshow("Anime Image", anime)
output_name = "anime_output.jpg"
cv2.imwrite(output_name, anime)

print(f" Saved as {output_name}")
cv2.waitKey(0)
cv2.destroyAllWindows()
input("Press Enter to exit...")